In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import ast
from slrcli.slr_post_analysis import splitable_columns

In [ ]:
def interpret_set(s):
    return ast.literal_eval(s)

very_relevant = pd.read_csv("data/generated/very_relevant.tsv", sep='\t')
very_relevant[splitable_columns] = very_relevant[splitable_columns].map(interpret_set)
included = pd.read_csv("data/generated/included.tsv", sep='\t')
included[splitable_columns] = included[splitable_columns].map(interpret_set)
with_non_music = pd.read_csv("data/generated/with_non_music.tsv", sep='\t')
with_non_music[splitable_columns] = with_non_music[splitable_columns].map(interpret_set)


post_hoc = pd.read_csv("data/annotated/post_hoc_2026-03-30.tsv", sep='\t')

print(very_relevant.shape)
print(included.shape)
print(with_non_music.shape)
print(post_hoc.shape)


In [ ]:
def filter_studies(df, *conditions):
    mask = np.ones(len(df), dtype=bool)
    for cond in conditions:
        mask = mask & cond(df)
    return df.loc[mask]

def set_contains_at_least_one_of(col, val_set):
    return lambda df: df[col].map(lambda s: bool(s & val_set))

def set_contains_all_of(col, val_set):
    return lambda df: df[col].map(lambda s: bool(val_set.issubset(s)))

def set_contains(col, val):
    return set_contains_at_least_one_of(col, {val})

def str_contains(col, val):
    return lambda df: df[col].map(lambda s: val.lower() in str(s).lower())


def style(df: pd.DataFrame, view_cols):
    return df[view_cols].sort_values(by="year").style\
        .set_properties(subset=['spatial attributes'], **{'width': '500px'})\
        .set_properties(subset=['authors'], **{'width': '200px', 'text-align': 'left'})\
        .set_properties(subset=['title'], **{'width': '300px', 'text-align': 'left'}) 

view_cols = ["title", "authors", "year", "reject", "downstream task", "audio model", "novel mechanism category", "spatial attributes"]

def flatten(xss):
    return [x for xs in xss for x in xs]

def attention(df: pd.DataFrame, col1: str, col2: str):
    pairs = df[[col1, col2]].apply(lambda row: list(itertools.product(row[col1], row[col2])), axis=1)
    return pd.Series(flatten(pairs)).value_counts()

def show(df, cols=view_cols):
    display(style(df, cols))

In [ ]:
# display(included)
d = filter_studies(
    included,
    # lambda df: df.year >= 2024,
    # set_contains("downstream task", "synthesis")
    set_contains_at_least_one_of("novel mechanism category", {"dl", "ml"})
)

print(d.shape)
display(style(d, view_cols + ['notes']))


In [ ]:
def graph_col(df, col, filter_values):
    val_year = df[[col, "year"]].explode(col)
    val_year = val_year.loc[val_year[col].isin(filter_values)]
    for task, group in val_year.groupby(col):
        line = group.value_counts().sort_index()
        x = line.index.map(lambda t: t[1])
        y = line.values
        plt.plot(x, y, label=task)
        plt.legend()

def graph_category(df):
    col = "novel mechanism category"
    cat_year = df[[col, "year"]].explode(col)
    cat_year.loc[cat_year[col] == "ml", col] = "mldl"
    cat_year.loc[cat_year[col] == "dl", col] = "mldl"
    dsp = cat_year[cat_year[col] == "dsp"].value_counts().sort_index()
    mldl = cat_year[cat_year[col] == "mldl"].value_counts().sort_index()
    print(mldl.index)
    plt.plot(dsp.index.map(lambda t: t[1]), dsp.values, label="dsp")
    plt.plot(mldl.index.map(lambda t: t[1]), mldl.values, label="ml/dl")

graph_col(included, "novel mechanism category", ["ml", "dl", "dsp"])
# graph_category(included)

In [ ]:
def mapper(s: str) -> str:
    mapping = {
        "ta_not_audio": "not_audio",
        "full_not_audio": "not_audio",
        "ta_no_spatial_analysis": "no_spatial_analysis",
        "ta_no_spatial_descriptor": "no_spatial_analysis",
        "m_full_no_spatial_descriptor": "no_spatial_analysis",
        "full_no_spatial_descriptor": "no_spatial_analysis",
        "ta_not_music": "not_music",
        "full_not_music": "not_music",
        "n": "no",
        "m": "unclear_too_old",
        "m_recent": "unclear",
        "ta_not_english": "not_english",
        "ta_english": "not_english",
        "full_not_english": "not_english",
    }
    return mapping.get(s, s)

def compare_post_hoc(df: pd.DataFrame):
    new_df = df.copy()

    # Map Richard's tags to the same text that Andreas was given in the dropdown menu
    new_df['reject'] = new_df["reject"].map(mapper)

    # Blank fields required deiscussion. Mark them as unclear.
    new_df.loc[new_df['Reject Andreas'].isna(), 'Reject Andreas'] = 'unclear'

    # Reject Andreas' maybes that are too old.
    new_df.loc[(new_df.year < 2020) & (new_df['Reject Andreas'] == 'unclear'), 'Reject Andreas'] = 'unclear_too_old'
    
    def decide_outcome(row: pd.Series):
        if (row['reject'] == 'no' and row['Reject Andreas'] == 'no') or (row['reject'] == 'unclear' and row['Reject Andreas'] == 'unclear'):
            return 'both_included'
        elif row['reject'] != 'no' and row['Reject Andreas'] == 'no':
            return "post_hoc_included"
        elif row['reject'] == 'no' and row['Reject Andreas'] != 'no':
            return "post_hoc_rejected"
        elif row['reject'] == row['Reject Andreas']:
            return "reject_same_reason"
        elif row['reject'] != row['Reject Andreas']:
            return "reject_different_reason"

    new_df["post_hoc_outcome"] = new_df.apply(decide_outcome, axis=1)

    return new_df


outcome = compare_post_hoc(post_hoc)
print(outcome.post_hoc_outcome.value_counts())



def show(df):
    post_hoc_cols = ['title', 'year', 'reject', 'Reject Andreas', 'Discussion']
    display(
        df[post_hoc_cols].sort_values(by='year').style\
            .set_properties(subset=['title'], **{'width': '300px', 'text-align': 'left'}) 
    )

show(outcome[outcome.post_hoc_outcome == 'reject_different_reason'])